In [1]:
import polars as pl
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool, TapTool, OpenURL
from bokeh.io import output_notebook
from bokeh.models import CategoricalColorMapper
from bokeh.palettes import Category10, Category20, Turbo256
from bokeh.transform import jitter
import numpy as np

In [2]:
data = pl.read_csv("unwatched_movie_data.csv", null_values="NULL")
data = data.drop_nulls().drop_nans()
data

tconst,actor,producer,composer,cinematographer,director,writer,genre,start_year,average_rating,num_votes
str,str,str,str,str,str,str,str,i64,f64,i64
"""tt0031381""","""nm0000022""","""nm0006388""","""nm0000070""","""nm0005735""","""nm0281808""","""nm0842485""","""Drama""",1939,8.2,355991
"""tt0031381""","""nm0000022""","""nm0006388""","""nm0000070""","""nm0005735""","""nm0281808""","""nm0842485""","""Romance""",1939,8.2,355991
"""tt0031381""","""nm0000022""","""nm0006388""","""nm0000070""","""nm0005735""","""nm0281808""","""nm0842485""","""War""",1939,8.2,355991
"""tt0031381""","""nm0000046""","""nm0006388""","""nm0000070""","""nm0005735""","""nm0281808""","""nm0842485""","""Drama""",1939,8.2,355991
"""tt0031381""","""nm0000046""","""nm0006388""","""nm0000070""","""nm0005735""","""nm0281808""","""nm0842485""","""Romance""",1939,8.2,355991
…,…,…,…,…,…,…,…,…,…,…
"""tt9873892""","""nm0000662""","""nm5533924""","""nm8882407""","""nm1092952""","""nm3541432""","""nm3541432""","""Sci-Fi""",2023,6.7,47810
"""tt9873892""","""nm0000662""","""nm5533924""","""nm8882407""","""nm1092952""","""nm3541432""","""nm3541432""","""Comedy""",2023,6.7,47810
"""tt9873892""","""nm4544635""","""nm5533924""","""nm8882407""","""nm1092952""","""nm3541432""","""nm3541432""","""Mystery""",2023,6.7,47810


In [3]:
# multi hot encoding
tconst_genre = data.select(
    pl.col("tconst"),
    pl.col("genre").cast(pl.Categorical)
).unique().to_dummies(columns=["genre"]).group_by("tconst").sum()

tconst_actor = data.select(
    pl.col("tconst"),
    pl.col("actor").cast(pl.Categorical)
).unique().to_dummies(columns=["actor"]).group_by("tconst").sum()

tconst_producer = data.select(
    pl.col("tconst"),
    pl.col("producer").cast(pl.Categorical)
).unique().to_dummies(columns=["producer"]).group_by("tconst").sum()

tconst_composer = data.select(
    pl.col("tconst"),
    pl.col("composer").cast(pl.Categorical)
).unique().to_dummies(columns=["composer"]).group_by("tconst").sum()

tconst_cinematographer = data.select(
    pl.col("tconst"),
    pl.col("cinematographer").cast(pl.Categorical)
).unique().to_dummies(columns=["cinematographer"]).group_by("tconst").sum()

tconst_director = data.select(
    pl.col("tconst"),
    pl.col("director").cast(pl.Categorical)
).unique().to_dummies(columns=["director"]).group_by("tconst").sum()

tconst_writer = data.select(
    pl.col("tconst"),
    pl.col("writer").cast(pl.Categorical)
).unique().to_dummies(columns=["writer"]).group_by("tconst").sum()


In [4]:
ml_data = (
    data.select(
        pl.col("tconst"),
        # pl.col("num_votes"), # this becomes second component
        # pl.col("average_rating") # this becomes first component
    )
    .unique()
    .join(tconst_genre, on="tconst", how="left")
    .join(tconst_actor, on="tconst", how="left")
    .join(tconst_producer, on="tconst", how="left")
    .join(tconst_composer, on="tconst", how="left")
    .join(tconst_cinematographer, on="tconst", how="left")
    .join(tconst_director, on="tconst", how="left")
    .join(tconst_writer, on="tconst", how="left")
)

labels = ml_data["tconst"]
ml_data = ml_data.drop("tconst")
ml_data

genre_Action,genre_Adventure,genre_Animation,genre_Biography,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Family,genre_Fantasy,genre_History,genre_Horror,genre_Music,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Short,genre_Sport,genre_Thriller,genre_War,genre_Western,actor_nm0000006,actor_nm0000007,actor_nm0000017,actor_nm0000018,actor_nm0000022,actor_nm0000023,actor_nm0000024,actor_nm0000027,actor_nm0000034,actor_nm0000044,actor_nm0000046,actor_nm0000048,actor_nm0000059,actor_nm0000060,actor_nm0000061,…,writer_nm7226510,writer_nm7264226,writer_nm7267981,writer_nm7328716,writer_nm7554519,writer_nm7698350,writer_nm7759605,writer_nm7974908,writer_nm8026647,writer_nm8047398,writer_nm8175340,writer_nm8213708,writer_nm8748334,writer_nm8828791,writer_nm8844932,writer_nm8921482,writer_nm8962876,writer_nm8971373,writer_nm8972520,writer_nm8990729,writer_nm9140637,writer_nm9276804,writer_nm9324003,writer_nm9360569,writer_nm9526824,writer_nm9526825,writer_nm9557552,writer_nm9557553,writer_nm9643171,writer_nm9701752,writer_nm9831079,writer_nm9847292,writer_nm9849369,writer_nm9878616,writer_nm9967372,writer_nm9967860,writer_nm9991653
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
pca = PCA(n_components=2)
ml_data_pca = pca.fit_transform(ml_data)
pcs = pl.from_numpy(ml_data_pca, schema=["pc1", "pc2"])
result = pl.concat([labels.to_frame(), pcs], how="horizontal")
result

tconst,pc1,pc2
str,f64,f64
"""tt0056172""",0.417189,0.185661
"""tt7279188""",-0.070119,0.858804
"""tt27695005""",-0.529649,0.737307
"""tt0395169""",0.912847,0.168483
"""tt0059527""",0.721573,0.06308
…,…,…
"""tt6791350""",-1.547368,0.472889
"""tt2103281""",-0.29554,-0.409874
"""tt15339456""",0.269425,0.909655


In [6]:
# nu de normale data erbij
normale_data = pl.read_csv("unwatched_movie_info.csv", null_values="NULL")
data = normale_data.join(result, how="left", on="tconst")
data

tconst,start_year,average_rating,num_votes,primary_title,priority,title_type,genres,directors,pc1,pc2
str,i64,f64,i64,str,bool,str,str,str,f64,f64
"""tt0015324""",1924,8.1,65521,"""Sherlock Jr.""",false,"""movie""","""Action - Comedy - Romance""","""Buster Keaton""",null,null
"""tt0017925""",1926,8.1,106053,"""The General""",false,"""movie""","""Action - Adventure - Comedy""","""Buster Keaton - Clyde Bruckman""",null,null
"""tt0021749""",1931,8.5,216100,"""City Lights""",true,"""movie""","""Comedy - Drama - Romance""","""Charles Chaplin""",null,null
"""tt0025316""",1934,8.1,119606,"""It Happened One Night""",false,"""movie""","""Comedy - Romance""","""Frank Capra""",null,null
"""tt0031381""",1939,8.2,355991,"""Gone with the Wind""",false,"""movie""","""Drama - Romance - War""","""Victor Fleming""",0.811466,0.174011
…,…,…,…,…,…,…,…,…,…,…
"""tt33764258""",2026,null,null,"""The Odyssey""",false,"""movie""","""Action - Adventure - Fantasy""","""Christopher Nolan""",null,null
"""tt32565993""",2026,null,null,"""The Sheep Detectives""",true,"""movie""","""Action - Comedy - Mystery""","""Kyle Balda""",null,null
"""tt15940132""",2026,6.3,85248,"""War Machine""",false,"""movie""","""Action - Sci-Fi - Thriller""","""Patrick Hughes""",-0.691055,-1.042142


In [7]:
output_notebook()

Loading BokehJS ...

In [8]:
source = ColumnDataSource(data.to_dict(as_series=False))

p = figure(width=900, height=600, title="Movie PCA Explorer")

p.circle("pc1", "pc2", size=8, alpha=0.6, source=source)

hover = HoverTool(tooltips=[
    ("Title", "@primary_title"),
    ("Score", "@average_rating"),
    ("Votes", "@num_votes"),
    ("Year", "@start_year"),
    ("Genres", "@genres"),
    ("Directors", "@directors"),
    ("IMDb", "@tconst"),
])
p.add_tools(hover)

show(p)

In [9]:
data = data.filter(pl.col("pc1").is_not_null())

In [10]:
db = KMeans(n_clusters=30, random_state=42)
groups = db.fit_predict(data[["pc1","pc2"]])

groups = pl.from_numpy(groups, schema=["group"])

# # pca = PCA(n_components=2)
# # ml_data_pca = pca.fit_transform(ml_data)
result = pl.concat([data, groups], how="horizontal")

In [11]:
# 1. Replace NULL groups with a label
result = result.with_columns(
    pl.col("group").fill_null("Unknown").alias("group"),
    ("https://www.imdb.com/title/" + pl.col("tconst") + "/").alias("url"),
    )

# 2. Convert to ColumnDataSource
source = ColumnDataSource(result.to_dict(as_series=False))

# 3. Extract unique groups
groups = sorted(result["group"].unique())

# 4. Build a large palette by sampling Turbo256
#    This gives you exactly len(groups) distinct colors
palette = [Turbo256[int(i)] for i in np.linspace(0, 255, len(groups))]

In [12]:
color_mapper = CategoricalColorMapper(factors=groups, palette=palette)

# 5. Plot with color mapping
p = figure(
    width=900,
    height=600,
    title="Movie PCA Explorer",
    tools="tap, box_zoom, wheel_zoom, reset, pan, hover, save",
)

p.circle(
    "pc1",
    "pc2",
    size=20,
    alpha=0.6,
    color={"field": "group", "transform": color_mapper},
    source=source,
)

hover = HoverTool(
    tooltips=[
        ("Title", "@primary_title"),
        ("Score", "@average_rating"),
        ("Votes", "@num_votes"),
        ("Year", "@start_year"),
        ("Genres", "@genres"),
        ("Directors", "@directors"),
        ("IMDb", "@tconst"),
    ]
)
p.select_one(TapTool).callback = OpenURL(url="@url")
p.add_tools(hover)


p.legend.click_policy = "hide"

show(p)


/tmp/ipykernel_1266524/3073401022.py:35: UserWarning: 
You are attempting to set `plot.legend.click_policy` on a plot that has zero legends added, this will have no effect.

Before legend properties can be set, you must add a Legend explicitly, or call a glyph method with a legend parameter set.

  p.legend.click_policy = "hide"


# clusterd mostly by genre combination

# todo: link for movie

In [13]:
result = result.with_columns(
    pl.col("group").fill_null("Unknown").alias("group")
)

source = ColumnDataSource(result.to_dict(as_series=False))

groups = sorted(result["group"].unique())
palette = Category20[len(groups)] if len(groups) <= 20 else Category20[20]

color_mapper = CategoricalColorMapper(
    factors=groups,
    palette=palette
)

p = figure(
    width=900,
    height=600,
    x_range=groups,
    title="Movies by Group"
)

p.circle(
    x="group",
    y=jitter("pc1", width=0.4),   # jitter on y-axis
    size=8,
    alpha=0.6,
    color={"field": "group", "transform": color_mapper},
    # legend_field="group",
    source=source
)

hover = HoverTool(tooltips=[
    ("Title", "@primary_title"),
    ("Score", "@average_rating"),
    ("Votes", "@num_votes"),
    ("Year", "@start_year"),
    ("Genres", "@genres"),
    ("Directors", "@directors"),
    ("IMDb", "@tconst"),
])
p.add_tools(hover)

p.legend.location = "top_left"
p.legend.click_policy = "hide"
show(p)

/tmp/ipykernel_1154421/2041205869.py:43: UserWarning: 
You are attempting to set `plot.legend.location` on a plot that has zero legends added, this will have no effect.

Before legend properties can be set, you must add a Legend explicitly, or call a glyph method with a legend parameter set.

  p.legend.location = "top_left"
/tmp/ipykernel_1154421/2041205869.py:44: UserWarning: 
You are attempting to set `plot.legend.click_policy` on a plot that has zero legends added, this will have no effect.

Before legend properties can be set, you must add a Legend explicitly, or call a glyph method with a legend parameter set.

  p.legend.click_policy = "hide"


# copilot improved, must test. contains bug

In [ ]:
import polars as pl
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from bokeh.io import output_notebook
from bokeh.plotting import figure, show
from bokeh.models import (
    ColumnDataSource, HoverTool, TapTool, OpenURL,
    CategoricalColorMapper
)
from bokeh.palettes import Turbo256

output_notebook()

# ---------------------------------------------------------
# 1. Load & clean data
# ---------------------------------------------------------
data = (
    pl.read_csv("unwatched_movie_data.csv", null_values="NULL")
      .drop_nulls()
      .drop_nans()
)


# ---------------------------------------------------------
# 2. Helper: multi-hot encode any categorical column
# ---------------------------------------------------------
def multi_hot(df: pl.DataFrame, col: str) -> pl.DataFrame:
    return (
        df.select("tconst", pl.col(col).cast(pl.Categorical))
          .unique()
          .to_dummies(columns=[col])
          .group_by("tconst")
          .sum()
    )


# ---------------------------------------------------------
# 3. Multi-hot encode all categorical features
# ---------------------------------------------------------
categorical_cols = [
    "genre", "actor", "producer", "composer",
    "cinematographer", "director", "writer"
]

encoded = [
    multi_hot(data, col)
    for col in categorical_cols
]

# Base ML table
ml_data = data.select("tconst").unique()

# Join all encoded tables
for enc in encoded:
    ml_data = ml_data.join(enc, on="tconst", how="left")


labels = ml_data["tconst"]
X = ml_data.drop("tconst")


# ---------------------------------------------------------
# 4. PCA
# ---------------------------------------------------------
pca = PCA(n_components=2)
pcs = pca.fit_transform(X)

pcs_df = pl.DataFrame(pcs, schema=["pc1", "pc2"])
pca_result = pl.concat([labels.to_frame(), pcs_df], how="horizontal")


# ---------------------------------------------------------
# 5. Merge PCA back into movie info
# ---------------------------------------------------------
info = pl.read_csv("unwatched_movie_info.csv", null_values="NULL")
data = info.join(pca_result, on="tconst", how="left")

data = data.drop_nulls().drop_nans()

# ---------------------------------------------------------
# 6. KMeans clustering
# ---------------------------------------------------------
kmeans = KMeans(n_clusters=30, random_state=42)
clusters = kmeans.fit_predict(data[["pc1", "pc2"]])

data = data.with_columns([
    pl.Series("group", clusters).cast(pl.Utf8),
    ("https://www.imdb.com/title/" + pl.col("tconst") + "/").alias("url")
])


# ---------------------------------------------------------
# 7. Bokeh visualization
# ---------------------------------------------------------
source = ColumnDataSource(data.to_dict(as_series=False))

groups = sorted(data["group"].unique())
groups = [str(g) for g in groups]
palette = [Turbo256[int(i)] for i in np.linspace(0, 255, len(groups))]
color_mapper = CategoricalColorMapper(factors=groups, palette=palette)

p = figure(
    width=900,
    height=600,
    title="Movie PCA Explorer",
    tools="tap,box_zoom,wheel_zoom,reset,pan,hover,save",
)

p.circle(
    "pc1", "pc2",
    size=20,
    alpha=0.6,
    color={"field": "group", "transform": color_mapper},
    source=source,
)

hover = HoverTool(tooltips=[
    ("Title", "@primary_title"),
    ("Score", "@average_rating"),
    ("Votes", "@num_votes"),
    ("Year", "@start_year"),
    ("Genres", "@genres"),
    ("Directors", "@directors"),
    ("IMDb", "@tconst"),
])

p.add_tools(hover)
p.select_one(TapTool).callback = OpenURL(url="@url")

show(p)


Loading BokehJS ...

ColumnNotFoundError: unable to find column "group"; valid columns: ["tconst", "start_year", "average_rating", "num_votes", "primary_title", "priority", "title_type", "genres", "directors", "pc1", "pc2"]